# 03 — Baselines

The simple benchmark every later model has to beat: TF-IDF text features
fed to Logistic Regression. Two tasks: predict the product, and predict escalation.

Plan:
- TF-IDF + Logistic Regression (use `class_weight="balanced"` for the rare labels)
- Score with macro-F1 and PR-AUC (not plain accuracy — the data is imbalanced)
- Log each run to MLflow

Why a baseline first? If a fancy transformer later scores 0.90 but this simple
model already scores 0.88, the transformer barely earned its extra cost.
This notebook puts a number on that.

In [25]:
import os
import pandas as pd

# Work from the project root so the data paths are correct.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

train = pd.read_parquet("data/processed/train.parquet")
test = pd.read_parquet("data/processed/test.parquet")
print("train:", len(train), " test:", len(test))

train: 47121  test: 34694


## What exactly are we predicting?

Two separate tasks, both using ONLY the complaint text:

1. **Product** (9 classes): which product is the complaint about — credit
   reporting, debt collection, mortgage, and so on. Useful for routing a
   complaint to the right team the moment it arrives.
2. **Monetary relief** (yes/no): did the company end up paying the consumer?
   Only ~3% of complaints end that way, so this is a needle-in-a-haystack task.

One label we are deliberately **not** modeling yet: `is_untimely` (the company
missed its response deadline). It stays very rare even at 6-month scale, so we
keep the focus on the product and monetary tasks. The reasoning lives in
`reports/target_decision_memo.md`.

In [26]:
# 1) Look at what we are predicting.

print("--- Product (what the complaint is about) ---")
print(train["product"].value_counts())
print()
print("--- Monetary relief (did the consumer get money back?) ---")
print("train positives:", train["is_monetary_relief"].sum(),
      "of", len(train), "=", round(100 * train["is_monetary_relief"].mean(), 2), "%")
print("test positives: ", test["is_monetary_relief"].sum(),
      "of", len(test), "=", round(100 * test["is_monetary_relief"].mean(), 2), "%")

--- Product (what the complaint is about) ---
product
Credit card or prepaid card                                                     8000
Debt collection                                                                 8000
Checking or savings account                                                     8000
Credit reporting, credit repair services, or other personal consumer reports    8000
Mortgage                                                                        5736
Money transfer, virtual currency, or money service                              3254
Vehicle loan or lease                                                           3156
Payday loan, title loan, or personal loan                                       1663
Student loan                                                                    1312
Name: count, dtype: int64

--- Monetary relief (did the consumer get money back?) ---
train positives: 3417 of 47121 = 7.25 %
test positives:  1041 of 34694 = 3.0 %


## Turning text into numbers: TF-IDF

TF-IDF gives every complaint one long row of numbers — one slot per word (or
word pair). A word scores high for a complaint if it appears often in THAT
complaint but is rare across the others. So "the" scores near zero everywhere,
while "repossession" lights up strongly in the few complaints that mention it.

We build two flavors and also try them combined:

- **word features** — single words and two-word phrases
- **character features** — chunks of 3–5 characters. These are robust to typos
  and cope well with the `XXXX` redaction masks that fill this dataset.

In [27]:
# 2) Turn the text into numbers. Fit on train only, then apply to test.
#    (Fitting on test would leak information about the future.)
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

train_text = train["consumer_complaint_narrative"]
test_text = test["consumer_complaint_narrative"]

# Word features: single words and 2-word phrases seen in at least 3 complaints.
word_tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3,
                             max_features=60000, sublinear_tf=True)
X_train_word = word_tfidf.fit_transform(train_text)
X_test_word = word_tfidf.transform(test_text)
print("word features:    ", X_train_word.shape)

# Character features: 3-to-5-character chunks, staying inside word boundaries.
char_tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3,
                             max_features=60000, sublinear_tf=True)
X_train_char = char_tfidf.fit_transform(train_text)
X_test_char = char_tfidf.transform(test_text)
print("char features:    ", X_train_char.shape)

# Both together, side by side.
X_train_both = hstack([X_train_word, X_train_char]).tocsr()
X_test_both = hstack([X_test_word, X_test_char]).tocsr()
print("word + char:      ", X_train_both.shape)

word features:     (47121, 60000)
char features:     (47121, 60000)
word + char:       (47121, 120000)


In [28]:
# 3) Set up MLflow so every run is recorded (settings + scores).
#    Runs land in the local mlruns/ folder. To browse them later, run
#    "mlflow ui" in a terminal (from the project root) and open localhost:5000.
import mlflow

mlflow.set_experiment("phase2_baselines_6mo")

# We also collect every score in these plain lists, to build a report at the end.
product_results = []
monetary_results = []

## Task A — which product is this complaint about?

Main score: **macro-F1**. That is the F1 score of each of the 9 classes,
averaged so every class counts EQUALLY. Why: 65% of complaints are credit
reporting, so a lazy model that always answers "credit reporting" is 65%
accurate — and useless. Macro-F1 punishes that laziness, because such a model
scores 0 on the eight other classes.

Accuracy is printed too, but only for reference.

In [29]:
# Task A, run 1: word features -> Logistic Regression.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report

model = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42)
model.fit(X_train_word, train["product"])
pred_product_word = model.predict(X_test_word)

macro_f1 = f1_score(test["product"], pred_product_word, average="macro")
accuracy = accuracy_score(test["product"], pred_product_word)
print("macro-F1:", round(macro_f1, 4), "  accuracy:", round(accuracy, 4))

with mlflow.start_run(run_name="product_logreg_word"):
    mlflow.log_param("task", "product")
    mlflow.log_param("model", "LogisticRegression (balanced)")
    mlflow.log_param("features", "word tf-idf 1-2 grams")
    mlflow.log_metric("macro_f1", macro_f1)
    mlflow.log_metric("accuracy", accuracy)

product_results.append(["LogisticRegression", "word", round(macro_f1, 4), round(accuracy, 4)])

macro-F1: 0.6972   accuracy: 0.8278


In [30]:
# Task A, run 2: word + char features -> Logistic Regression.
model = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42)
model.fit(X_train_both, train["product"])
pred_product_both = model.predict(X_test_both)

macro_f1 = f1_score(test["product"], pred_product_both, average="macro")
accuracy = accuracy_score(test["product"], pred_product_both)
print("macro-F1:", round(macro_f1, 4), "  accuracy:", round(accuracy, 4))

with mlflow.start_run(run_name="product_logreg_wordchar"):
    mlflow.log_param("task", "product")
    mlflow.log_param("model", "LogisticRegression (balanced)")
    mlflow.log_param("features", "word + char tf-idf")
    mlflow.log_metric("macro_f1", macro_f1)
    mlflow.log_metric("accuracy", accuracy)

product_results.append(["LogisticRegression", "word + char", round(macro_f1, 4), round(accuracy, 4)])

macro-F1: 0.7065   accuracy: 0.8337


In [31]:
# Task A, run 3: word + char features -> LinearSVC (a close cousin of
# logistic regression that often edges it out on text).
from sklearn.svm import LinearSVC

model = LinearSVC(class_weight="balanced", dual="auto", random_state=42)
model.fit(X_train_both, train["product"])
pred_product_svc = model.predict(X_test_both)

macro_f1 = f1_score(test["product"], pred_product_svc, average="macro")
accuracy = accuracy_score(test["product"], pred_product_svc)
print("macro-F1:", round(macro_f1, 4), "  accuracy:", round(accuracy, 4))

with mlflow.start_run(run_name="product_linearsvc_wordchar"):
    mlflow.log_param("task", "product")
    mlflow.log_param("model", "LinearSVC (balanced)")
    mlflow.log_param("features", "word + char tf-idf")
    mlflow.log_metric("macro_f1", macro_f1)
    mlflow.log_metric("accuracy", accuracy)

product_results.append(["LinearSVC", "word + char", round(macro_f1, 4), round(accuracy, 4)])

macro-F1: 0.7025   accuracy: 0.8295


In [32]:
# Per-class detail for the best product run above (run 2: Logistic Regression
# on word + char features — the best macro-F1).
# Rare products are much harder — macro-F1 feels that pain, accuracy hides it.
print(classification_report(test["product"], pred_product_both,
                            digits=3, zero_division=0))

                                                                              precision    recall  f1-score   support

                                                 Checking or savings account      0.798     0.780     0.789      2590
                                                 Credit card or prepaid card      0.680     0.805     0.737      2863
Credit reporting, credit repair services, or other personal consumer reports      0.977     0.850     0.909     22418
                                                             Debt collection      0.624     0.809     0.704      3427
                          Money transfer, virtual currency, or money service      0.561     0.767     0.648       836
                                                                    Mortgage      0.846     0.931     0.886      1139
                                   Payday loan, title loan, or personal loan      0.384     0.686     0.492       488
                                                       

## Task B — will this complaint end with the company paying money?

Only ~3 in 100 complaints end in monetary relief, so "always say no" is ~97%
accurate and completely useless. Main score: **PR-AUC** (area under the
precision–recall curve). Plain meaning: sort all complaints by how confident
the model is that money will be paid, then walk down that list — PR-AUC is high
when the real payouts cluster near the top. A random guesser lands at the
positive rate, ≈ 0.03 here. That is the floor to compare against.

We also print precision / recall / F1 for the positive class at the default
cut-off, as one concrete operating point.

In [33]:
# Task B, run 1: word features -> Logistic Regression.
from sklearn.metrics import average_precision_score, roc_auc_score

y_train = train["is_monetary_relief"]
y_test = test["is_monetary_relief"]

model = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42)
model.fit(X_train_word, y_train)

scores = model.predict_proba(X_test_word)[:, 1]   # confidence that money is paid
pr_auc = average_precision_score(y_test, scores)
roc_auc = roc_auc_score(y_test, scores)
f1_pos = f1_score(y_test, model.predict(X_test_word))
print("PR-AUC:", round(pr_auc, 4), "  ROC-AUC:", round(roc_auc, 4),
      "  F1 (positive class):", round(f1_pos, 4))

with mlflow.start_run(run_name="monetary_logreg_word"):
    mlflow.log_param("task", "monetary_relief")
    mlflow.log_param("model", "LogisticRegression (balanced)")
    mlflow.log_param("features", "word tf-idf 1-2 grams")
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("f1_positive", f1_pos)

monetary_results.append(["LogisticRegression", "word", round(pr_auc, 4), round(roc_auc, 4), round(f1_pos, 4)])

PR-AUC: 0.3333   ROC-AUC: 0.9399   F1 (positive class): 0.3706


In [34]:
# Task B, run 2: word + char features -> Logistic Regression.
model = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42)
model.fit(X_train_both, y_train)

scores = model.predict_proba(X_test_both)[:, 1]
pr_auc = average_precision_score(y_test, scores)
roc_auc = roc_auc_score(y_test, scores)
f1_pos = f1_score(y_test, model.predict(X_test_both))
print("PR-AUC:", round(pr_auc, 4), "  ROC-AUC:", round(roc_auc, 4),
      "  F1 (positive class):", round(f1_pos, 4))

with mlflow.start_run(run_name="monetary_logreg_wordchar"):
    mlflow.log_param("task", "monetary_relief")
    mlflow.log_param("model", "LogisticRegression (balanced)")
    mlflow.log_param("features", "word + char tf-idf")
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("f1_positive", f1_pos)

monetary_results.append(["LogisticRegression", "word + char", round(pr_auc, 4), round(roc_auc, 4), round(f1_pos, 4)])

PR-AUC: 0.3337   ROC-AUC: 0.9396   F1 (positive class): 0.372


In [35]:
# Task B, run 3: word + char features -> LinearSVC.
# LinearSVC has no predict_proba; its decision_function still ranks complaints,
# which is all PR-AUC needs.
model = LinearSVC(class_weight="balanced", dual="auto", random_state=42)
model.fit(X_train_both, y_train)

scores = model.decision_function(X_test_both)
pr_auc = average_precision_score(y_test, scores)
roc_auc = roc_auc_score(y_test, scores)
f1_pos = f1_score(y_test, model.predict(X_test_both))
print("PR-AUC:", round(pr_auc, 4), "  ROC-AUC:", round(roc_auc, 4),
      "  F1 (positive class):", round(f1_pos, 4))

with mlflow.start_run(run_name="monetary_linearsvc_wordchar"):
    mlflow.log_param("task", "monetary_relief")
    mlflow.log_param("model", "LinearSVC (balanced)")
    mlflow.log_param("features", "word + char tf-idf")
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("f1_positive", f1_pos)

monetary_results.append(["LinearSVC", "word + char", round(pr_auc, 4), round(roc_auc, 4), round(f1_pos, 4)])

PR-AUC: 0.2769   ROC-AUC: 0.9093   F1 (positive class): 0.3422


In [36]:
# 4) Put every score in two small tables and save them as a report.
import datetime

product_table = pd.DataFrame(product_results,
                             columns=["model", "features", "macro_f1", "accuracy"])
monetary_table = pd.DataFrame(monetary_results,
                              columns=["model", "features", "pr_auc", "roc_auc", "f1_positive"])
print("Task A — product")
print(product_table.to_string(index=False))
print()
print("Task B — monetary relief")
print(monetary_table.to_string(index=False))

# The single number each later model must beat, per task.
best_product = product_table.loc[product_table["macro_f1"].idxmax()]
best_monetary = monetary_table.loc[monetary_table["pr_auc"].idxmax()]

train_dates = pd.to_datetime(train["date_received"])
test_dates = pd.to_datetime(test["date_received"])

lines = []
lines.append("# Phase 2 — Baseline results")
lines.append("")
lines.append("Generated by `notebooks/03_baseline.ipynb` on " + str(datetime.date.today()) + ".")
lines.append("")
lines.append("Data: 6-month set (Jan-Jun 2023), TRAIN class-balanced in notebook 01, TEST left natural. Train " + str(len(train)) +
             " rows (" + str(train_dates.min().date()) + " to " + str(train_dates.max().date()) + "), " +
             "test " + str(len(test)) +
             " rows (" + str(test_dates.min().date()) + " to " + str(test_dates.max().date()) + "). " +
             "Temporal split, duplicates removed, leakage columns dropped in notebook 01.")
lines.append("")

lines.append("## Task A — product classification (9 classes, macro-F1)")
lines.append("")
lines.append("```")
lines.append(product_table.to_string(index=False))
lines.append("```")
lines.append("")

lines.append("## Task B — monetary relief (" +
             str(round(100 * y_test.mean(), 2)) + "% positive in test, PR-AUC)")
lines.append("")
lines.append("```")
lines.append(monetary_table.to_string(index=False))
lines.append("```")
lines.append("")

lines.append("## The numbers the Phase 3 transformer must beat")
lines.append("")
lines.append("- Product macro-F1: **" + str(best_product["macro_f1"]) +
             "** (" + best_product["model"] + ", " + best_product["features"] + ")")
lines.append("- Monetary-relief PR-AUC: **" + str(best_monetary["pr_auc"]) +
             "** (" + best_monetary["model"] + ", " + best_monetary["features"] + ")")
lines.append("")

lines.append("## Notes")
lines.append("")
lines.append("- Accuracy is shown only for reference; the data is heavily imbalanced.")
lines.append("- Random-guess floor for PR-AUC is the positive rate, about " +
             str(round(y_test.mean(), 3)) + ".")
lines.append("- `is_untimely` is not modeled: only " + str(int(train["is_untimely"].sum())) +
             " positive train examples at March scale. Revisit on the 6-month data.")
lines.append("- All runs are logged in MLflow (mlruns/), experiment `phase2_baselines_6mo`.")

with open("reports/baseline_report.md", "w") as f:
    f.write("\n".join(lines) + "\n")
print()
print("Saved reports/baseline_report.md")

Task A — product
             model    features  macro_f1  accuracy
LogisticRegression        word    0.6972    0.8278
LogisticRegression word + char    0.7065    0.8337
         LinearSVC word + char    0.7025    0.8295

Task B — monetary relief
             model    features  pr_auc  roc_auc  f1_positive
LogisticRegression        word  0.3333   0.9399       0.3706
LogisticRegression word + char  0.3337   0.9396       0.3720
         LinearSVC word + char  0.2769   0.9093       0.3422

Saved reports/baseline_report.md


## Takeaways

This is the baseline on the **6-month, class-balanced training set** (test left at
its natural mix). The exact numbers are in the tables above and in
`reports/baseline_report.md`.

- **Product routing.** The best macro-F1 is the number the transformer (notebook
  04) has to clearly beat to justify its extra cost. The earlier March-only
  baseline scored 0.7316 macro-F1 on its own, smaller test set.
- **Rare products are still the weak spot.** The per-class report shows which
  classes remain hardest; that is where a transformer would have to help.
- **Metric choice changes the winner.** LinearSVC can win on *accuracy* but not
  *macro-F1*, because it leans on the big classes. Report macro-F1.
- **Monetary relief is hard** (~3% positive). PR-AUC is the score to watch;
  ROC-AUC looks good for every model under heavy imbalance. Char n-grams help
  product more than monetary, so feature choices are task-dependent.
- Every run is in MLflow under experiment `phase2_baselines_6mo`.
